## youtube shorts review

In [3]:
# Chap 16. 개인프로젝트 유튜브 쇼츠 리뷰 다중 수집기

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service # (더 이상 필요 없지만 호환성을 위해 유지)
import time
import math
import pandas as pd
import random
import os
import re
import pymysql  # 📌 DB 연동을 위해 추가된 모듈

print("=" *80)
print(" 개인프로젝트 유튜브 리뷰 (쇼츠 다중 영상 수집)")
print("=" *80)
print("\n")

query_txt = input('1.크롤링할 유튜브의 키워드는 무엇입니까?: ')
query_txt = query_txt.replace('"','')

# 🌟 수정됨: 영상 1개 당 수집할 건수로 기준 변경
cnt = int(input('2.쇼츠 "영상 1개 당" 수집할 댓글은 몇 건입니까?(예: 10): '))
page_cnt = math.ceil(cnt)

target_video_cnt = int(input('3.탐색할 쇼츠 영상의 개수는 총 몇 개입니까?(예: 3): '))

f_dir = input("4.파일을 저장할 폴더명만 쓰세요(예:c:\\py_temp\\):")
if f_dir=='' :
    f_dir='c:\\py_temp\\결과 추출 - 리뷰, 댓글\\'

print("요청하신 데이터를 수집하고 있으니 잠시만 기다려 주세요~~")

# 폴더 및 파일 생성
n = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (n.tm_year, n.tm_mon, n.tm_mday, n.tm_hour, n.tm_min, n.tm_sec)

os.makedirs(f_dir+s+'-'+query_txt, exist_ok=True)
os.chdir(f_dir+s+'-'+query_txt)

ff_name=f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.txt'
fc_name=f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.csv'
fx_name=f_dir+s+'-'+query_txt+'\\'+s+'-'+query_txt+'.xls'

# ==========================================================
# 🌟 핵심 수정 부분: 크롬 드라이버 자동 스캔 및 실행
# ==========================================================
s_time = time.time( )
# 기존의 로컬 경로 지정 코드는 삭제했습니다.
# s = Service("c:/py_temp/chromedriver.exe") 

# 아무 경로 없이 실행하면 셀레니움 내장 매니저가 내 PC의 크롬을 자동 연동합니다.
driver = webdriver.Chrome()

driver.get("https://www.youtube.com/")
driver.maximize_window()
time.sleep(4)

# 유튜브 검색
keyword = driver.find_element(By.XPATH,"//*[@id='center']/yt-searchbox/div[1]/div/form/input")
for a in query_txt :
    keyword.send_keys(a)
    time.sleep(0.3)
time.sleep(1) 
driver.find_element(By.XPATH,'//*[@id="center"]/yt-searchbox/div[1]/button/span/span/div').click()
time.sleep(random.randrange(3,5))

try:
    # 필터/쇼츠 탭 클릭
    driver.find_element(By.XPATH,'//*[@id="chips"]/yt-chip-cloud-chip-renderer[2]').click()
    time.sleep(random.randrange(2,4))
except:
    pass

# 누적 리스트 및 전체 카운트
y_URL2=[] 
reviewer2=[] 
review_d2=[] 
review2=[] 
like2=[] 
total_count = 0  # 🌟 새로 추가됨: 모든 영상 통합 누적 수집 건수

for i in range(target_video_cnt):
    print(f"\n{'=' * 80}")
    print(f"[{i+1}번째 유튜브 영상으로 진입합니다]")
    
    videos = driver.find_elements(By.ID, "thumbnail")
    valid_videos = [v for v in videos if v.get_attribute("href")]
    
    if i >= len(valid_videos):
        print("더 이상 클릭할 영상이 부족합니다.")
        break
        
    v = valid_videos[i]  
    href = v.get_attribute("href")
    print("클릭할 URL:", href)
    
    # 일반 click() 시 화면에 안 보이면 에러날 수 있어 자바스크립트 우회 클릭 사용
    driver.execute_script("arguments[0].click();", v)
    
    time.sleep(5)  
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    comment_btn = soup.select_one('button[aria-label*="댓글"]')
    if comment_btn:
        parent_label = comment_btn.find_parent('label')
        result_span = parent_label.select_one('div.yt-spec-button-shape-with-label__label span') if parent_label else None
        if result_span:
            review_count_str = result_span.get_text(strip=True).replace(",", "")
            result4 = re.search(r"\d+", review_count_str)
            search_cnt = int(result4.group()) if result4 else 0
        else:
            search_cnt = 0
    else:
        search_cnt = 0
        
    print(f"👉 이 영상의 전체 댓글 건수 : {search_cnt} 건")

    try:
        driver.find_element(By.CSS_SELECTOR, 'button[aria-label*="댓글"]').click()
        time.sleep(4)
    except:
        print("댓글창을 여는데 실패했거나 이미 열려있습니다.")

    
    # 🌟 수정됨: 현재 영상에서 수집한 개수를 세는 변수 따로 생성
    video_comment_count = 0  
    prev_count = -1
    
    for a in range(1, page_cnt+1):
        print('리뷰 정보를 수집합니다. 잠시만 기다려 주세요~~~~~~~~')
        f = open(ff_name, 'a', encoding='UTF-8')

        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')
        y_URL = driver.current_url

        reple_result = soup.find_all('ytd-comment-thread-renderer')

        for li in reple_result:
            main_comment = li.find('ytd-comment-view-model')
            if not main_comment:
                continue

            video_comment_count += 1
            total_count += 1
            print("\n")
            print(f"🚀 [영상 {i+1}] 목표 총 {cnt}건 중 {video_comment_count}번째 댓글 수집 중 =========")

            f.write("\n")
            f.write(f"[{i+1}번째 영상] 총 {cnt} 건 중 {video_comment_count} 번째 리뷰====\n")
            f.write("1.동영상 URL: " + y_URL + "\n")
            y_URL2.append(y_URL)
            time.sleep(0.5)

            try:
                reviewer = li.find('div', id='header-author').find('a', 'yt-simple-endpoint style-scope ytd-comment-view-model').get_text()
            except:
                reviewer = "알 수 없음"
            f.write("2.댓글작성자명: " + reviewer.replace("\n", "").strip() + "\n")
            reviewer2.append(reviewer)
            
            try:
                review_d = li.find('div', id='header-author').find('span', id='published-time-text').get_text()
            except:
                review_d = "알 수 없음"
            f.write("3.댓글작성일자: " + review_d.replace("\n", "").strip() + "\n")
            review_d2.append(review_d)

            try:
                review = li.find('div', id='content').find('span', 'yt-core-attributed-string yt-core-attributed-string--white-space-pre-wrap').get_text()
            except:
                review = ""
            f.write("4.리뷰내용: " + review + "\n")
            review2.append(review)

            try:
                like_node = li.select_one('like-button-view-model span[role="text"]')
                if like_node:
                    like = like_node.get_text(strip=True)
                else:
                    backup_node = li.select_one('#vote-count-middle')
                    like = backup_node.get_text(strip=True) if backup_node else '0'
                if not like:
                    like = '0'
            except:
                like = '0'
            f.write("5.좋아요횟수:" + like + "\n")
            like2.append(like)

            # 🛑 1개 영상에 대한 목표 건수 달성 시 내부 수집 중단
            if video_comment_count >= cnt:
                print(f"\n✅ 이번 영상에서의 수집 목표량({cnt}건) 달성 완료!")
                break

        time.sleep(0.2)
        f.close()

        # 더 이상 수집되는게(스크롤 효과) 없거나 목표치 넘기면 탈출
        if video_comment_count == prev_count or video_comment_count >= cnt:
            break
        prev_count = video_comment_count

    # 4. 수집 완료 후 뒤로가기로 검색결과 화면 복귀
    print(f"\n[{i+1}번째 영상 댓글 수집 완료. 다음 영상을 찾기 위해 뒤로가기 합니다.]")
    driver.back()
    time.sleep(5)  # 검색결과 창 로딩을 넉넉히 대기


#Step 7. xls 형태와 csv 형태로 저장하기
news_reple = pd.DataFrame()
news_reple['동영상 URL']=pd.Series(y_URL2)
news_reple['댓글작성자명']=pd.Series(reviewer2)
news_reple['댓글 작성일자']=pd.Series(review_d2)
news_reple['리뷰내용']=pd.Series(review2)
news_reple['좋아요횟수']=pd.Series(like2)

news_reple.to_csv(fc_name,encoding="utf-8-sig",index=True)
news_reple.to_excel(fx_name ,index=True , engine='openpyxl')


# ==========================================================
# 📌 추가된 Step: 수집한 유튜브 정보 MySQL DB에 넣기
# ==========================================================
try:
    conn = pymysql.connect(
        host='localhost',        
        user='root',              
        password='Jx03151616~~',  
        db='youtube_db',  
        charset='utf8mb4',        
        cursorclass=pymysql.cursors.DictCursor
    )
    with conn.cursor() as cursor:
        
        # 테이블 생성 
        create_table_sql = """
        CREATE TABLE IF NOT EXISTS youtube_shorts_reviews (
            id INT AUTO_INCREMENT PRIMARY KEY, 
            video_url VARCHAR(255),       
            reviewer_name VARCHAR(255),         
            review_date VARCHAR(255),                 
            review_content TEXT,
            likes VARCHAR(50)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
        """
        cursor.execute(create_table_sql)
        
        # 데이터 삽입 
        insert_sql = """
        INSERT INTO youtube_shorts_reviews (video_url, reviewer_name, review_date, review_content, likes) 
        VALUES (%s, %s, %s, %s, %s)
        """
        for i in range(len(y_URL2)):
            cursor.execute(insert_sql, (
                y_URL2[i],              
                reviewer2[i],        
                review_d2[i], 
                review2[i], 
                like2[i]
            ))
        conn.commit()
        print("\n🎉 짝짝짝! 유튜브 쇼츠 리뷰 데이터 DB 저장까지 완벽하게 완료되었습니다!")
        
except Exception as e:
    print(f"\n[DB 에러] DB 저장 중 에러가 발생했습니다: {e}")
finally:
    try:
        conn.close()
    except:
        pass


# Step 8. 요약 정보 출력하기
e_time = time.time( )
t_time = e_time - s_time

print("\n")
print("=" *120)
print(f"1.모든 작업 종료. 수집된 전체(누적) 리뷰수는 {total_count} 건 입니다.")
print("2.총 소요시간은 %s 초 입니다 " %round(t_time,1))
print("3.파일 저장 완료: txt 파일명 : %s " %ff_name)
print("4.파일 저장 완료: csv 파일명 : %s " %fc_name)
print("5.파일 저장 완료: xls 파일명 : %s " %fx_name)
print("=" *120)

driver.close()

 개인프로젝트 유튜브 리뷰 (쇼츠 다중 영상 수집)


요청하신 데이터를 수집하고 있으니 잠시만 기다려 주세요~~

[1번째 유튜브 영상으로 진입합니다]
클릭할 URL: https://www.youtube.com/shorts/OVUfCH3oT6I
👉 이 영상의 전체 댓글 건수 : 55 건
리뷰 정보를 수집합니다. 잠시만 기다려 주세요~~~~~~~~


🚀 [영상 1] 목표 총 30건 중 1번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 2번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 3번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 4번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 5번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 6번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 7번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 8번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 9번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 10번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 11번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 12번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 13번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 14번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 15번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 16번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 17번째 댓글 수집 중 =========


🚀 [영상 1] 목표 총 30건 중 18번째 댓글 수집 중 =========


🚀